# Live CAMB + CosmoRec comparison: Capse.jl and jaxcapse

This notebook evaluates **TT, TE, EE, BB and PP** from the published 500k
emulators and a live CAMB 2.0.4 + CosmoRec call using the same worker and
settings used for generation. Outputs are lensed $D_\ell$ in their public
units. The metric tables report pointwise fractional errors where meaningful
and full-sky, noise-free cosmic-variance residuals at the editable cosmology
below. The companion Python notebook contains the spectrum and residual plots.

The notebook environment is `notebooks/Project.toml`. The sibling `jaxcapse`
and `emulator-zoo` checkouts and the local CAMB 2.0.4 + CosmoRec build are
required. Set `JAXCAPSE_PYTHON`, `JAXCAPSE_ROOT` or `CAMB_COSMOREC_ROOT` to
override local paths.


In [1]:
# EDIT THIS CELL to inspect another point in the trained domain.
parameter_names = ("ln10As", "ns", "tau", "H0", "omega_b", "omega_c", "Mnu", "w0", "wa")
lower_bounds = [2.5, 0.85, 0.02, 50.0, 0.02, 0.08, 0.0, -3.0, -3.0]
upper_bounds = [3.5, 1.05, 0.15, 90.0, 0.025, 0.16, 0.5, 0.5, 2.0]
params = Float64[3.044, 0.965, 0.054, 67.4, 0.02237, 0.12, 0.06, -1.0, 0.0]

length(params) == length(parameter_names) || error("Expected nine cosmological parameters")
all((lower_bounds .<= params) .& (params .<= upper_bounds)) ||
    error("Parameters are outside the published training bounds")
params[8] + params[9] < -0.5 || error("The training domain requires w0 + wa < -0.5")
params[7] == 0 && @warn "Exact Mnu=0 was not represented in the released training sample"

println("Parameters in training order:")
for i in eachindex(params)
    println("  ", parameter_names[i], " = ", params[i])
end

Parameters in training order:
  ln10As

 = 3.044
  ns = 0.965
  tau = 0.054
  H0 = 67.4
  omega_b = 0.02237
  omega_c = 0.12
  Mnu = 0.06
  w0 = -1.0
  wa = 0.0


In [2]:
using Pkg

notebooks_dir = isfile(joinpath(pwd(), "notebooks", "Project.toml")) ?
    joinpath(pwd(), "notebooks") : pwd()
@assert isfile(joinpath(notebooks_dir, "Project.toml")) "Start Jupyter from the Capse.jl checkout root or notebooks/"
Pkg.activate(notebooks_dir)

workspace = dirname(dirname(notebooks_dir))
jaxcapse_root = get(ENV, "JAXCAPSE_ROOT", joinpath(workspace, "jaxcapse"))
poetry_envs = joinpath(homedir(), ".cache", "pypoetry", "virtualenvs")
jaxcapse_python = get(ENV, "JAXCAPSE_PYTHON", "")
if isempty(jaxcapse_python)
    candidates = String[]
    for name in (isdir(poetry_envs) ? readdir(poetry_envs) : String[])
        startswith(name, "jaxcapse-") || continue
        python = joinpath(poetry_envs, name, "bin", "python")
        isfile(python) || continue
        works = success(pipeline(`$python -c "import jax, matplotlib, nbformat, ipykernel"`; stdout=devnull, stderr=devnull))
        works && push!(candidates, python)
    end
    length(candidates) == 1 || error("Set JAXCAPSE_PYTHON to the active jaxcapse Poetry interpreter.")
    jaxcapse_python = only(candidates)
end
@assert isfile(jaxcapse_python) jaxcapse_python
ENV["JULIA_CONDAPKG_BACKEND"] = "Null"
ENV["JULIA_PYTHONCALL_EXE"] = jaxcapse_python
ENV["JAX_ENABLE_X64"] = "true"
Pkg.instantiate()
println("PythonCall configured for the installed jaxcapse notebook environment")

  Activating 

project at `~/Desktop/work/CosmologicalEmulators/cmbcheb_test/Capse.jl/notebooks`


PythonCall configured for the installed jaxcapse notebook environment


In [3]:
using Capse, PythonCall, Printf, Statistics

spectra = ("TT", "TE", "EE", "BB", "PP")
models = Capse.trained_emulators["CAMB_MNUW0WACDM"]
ell = collect(Int, Capse.get_ℓgrid(models["TT"]))
@assert ell == collect(2:9500)
D_capse = Dict(name => Capse.get_Cℓ(params, models[name]) for name in spectra)
@assert all(values -> length(values) == length(ell) && all(isfinite, values), values(D_capse))

In [4]:
# Resolve the exact CAMB worker and recombination build used to generate the artifact.
camb_root = get(ENV, "CAMB_COSMOREC_ROOT", joinpath(workspace, "tools", "CAMB-cosmorec"))
worker_root = joinpath(workspace, "emulator-zoo", "Capse.jl", "camb_mnuw0wacdm")
@assert isfile(joinpath(camb_root, "camb", "camblib.so")) camb_root
@assert isfile(joinpath(worker_root, "camb_worker.py")) worker_root
sys = pyimport("sys")
for path in (camb_root, worker_root, jaxcapse_root)
    path in pyconvert(Vector{String}, sys.path) || sys.path.insert(0, path)
end
jax = pyimport("jax")
jax.config.update("jax_enable_x64", true)
jaxcapse = pyimport("jaxcapse")
worker = pyimport("camb_worker")
configuration = worker.backend_configuration()
camb_version = pyconvert(String, configuration["camb_version"])
recombination_model = pyconvert(String, configuration["recombination_model"])
@assert camb_version == "2.0.4"
@assert recombination_model == "CosmoRec"
println("Reference CAMB $(camb_version) / $(recombination_model)")


Reference CAMB 2.0.4 / CosmoRec


jaxcapse: Loading emulators into memory...
  camb_mnuw0wacdm: Loaded 5/5 emulators


In [5]:
jnp = pyimport("jax.numpy")
np = pyimport("numpy")
python_params = jnp.array(pylist(params), dtype=jnp.float64)
D_jaxcapse = Dict{String,Vector{Float64}}()
for name in spectra
    emulator = jaxcapse.trained_emulators["camb_mnuw0wacdm"][name]
    jax_ell = pyconvert(Vector{Int}, np.asarray(emulator.get_ell_grid()))
    @assert jax_ell == ell
    D_jaxcapse[name] = pyconvert(Vector{Float64}, np.asarray(emulator.get_Cl(python_params)))
end

camb_params = pydict(Dict(zip(parameter_names, params)))
camb_result = worker.compute_spectra(camb_params, 9500)
D_camb = Dict(name => pyconvert(Vector{Float64}, camb_result[name * "_dense"]) for name in spectra)
for name in spectra, values in (D_camb[name], D_capse[name], D_jaxcapse[name])
    @assert length(values) == length(ell) && all(isfinite, values)
end

pointwise_ranges = (("2-30", 2, 30), ("30-500", 30, 500), ("500-2000", 500, 2000),
                    ("2000-3000", 2000, 3000), ("3000-5000", 3000, 5000), ("5000-9500", 5000, 9500))

function print_residual_table(label, prediction)
    println("\n$(label) vs live CAMB (Dℓ ratios; full sky, no noise, Δℓ=1)")
    println("spec range        max frac      median frac      max |ΔD|/σCV   median |ΔD|/σCV")
    for name in spectra
        reference = D_camb[name]
        sigma = if name == "TE"
            sqrt.((D_camb["TT"] .* D_camb["EE"] .+ reference .^ 2) ./ (2 .* ell .+ 1))
        else
            sqrt.(2 ./ (2 .* ell .+ 1)) .* abs.(reference)
        end
        difference = abs.(prediction[name] .- reference)
        fractional = name == "TE" ? nothing : difference ./ abs.(reference)
        cosmic_variance = difference ./ sigma
        for (range_name, lo, hi) in pointwise_ranges
            selected = (ell .>= lo) .& (ell .<= hi)
            frac_text = if isnothing(fractional)
                "       n/a            n/a"
            else
                @sprintf("%12.3e %15.3e", maximum(fractional[selected]), median(fractional[selected]))
            end
            @printf("%-4s %-10s %s %15.3e %18.3e\n", name, range_name, frac_text,
                    maximum(cosmic_variance[selected]), median(cosmic_variance[selected]))
        end
    end
end

print_residual_table("Capse.jl", D_capse)
print_residual_table("jaxcapse", D_jaxcapse)
ell200_index = findfirst(==(200), ell)
@assert !isnothing(ell200_index)
@printf("\nAt ℓ=200: CAMB %.8g, Capse.jl %.8g, jaxcapse %.8g μK²\n",
        D_camb["TT"][ell200_index], D_capse["TT"][ell200_index], D_jaxcapse["TT"][ell200_index])
for name in spectra
    @printf("Max |Capse.jl - jaxcapse| in %s = %.3e\n", name,
            maximum(abs.(D_capse[name] .- D_jaxcapse[name])))
end


 || You are entering CosmoRec v3.0 beta (batch mode). Several initializations will be executed now.

 load_rates::Loading effective Rate data (HI) 
 resolved level # 0 :: 2 0 HI_index= 1
 resolved level # 1 :: 2 1 HI_index= 2
 resolved level # 2 :: 3 0 HI_index= 3
 resolved level # 3 :: 3 1 HI_index= 4
 resolved level # 4 :: 3 2 HI_index= 5
 load_rates::finished. # of resolved states: 5 Tg-points: 500


 load_rates::Loading effective Rate data (HeI) 
 resolved level # 0 :: 2 0 s= 0 j= 0 HeI_index= 1
 resolved level # 1 :: 2 1 s= 0 j= 1 HeI_index= 2
 resolved level # 2 :: 2 0 s= 1 j= 1 HeI_index= 3
 resolved level # 3 :: 2 1 s= 1 j= 1 HeI_index= 5
 load_rates::finished. # of resolved states: 4 Tg-points: 500

 Load_fcorr:: Loading correction factor for DPesc due to HI :
 /home/marcobonici/Desktop/work/CosmologicalEmulators/cmbcheb_test/tools/CosmoRec/./Development/Recombination/Data.fcorr/f.corr.dat
 Load_fcorr:: Number of redshift points: 40


 || The first run was successful. Will no